# wikisource-extract

[German Wikisource](https://de.wikisource.org/wiki/Heliand) reproduces [Behaghel's edition](https://archive.org/details/heliandundgenesi00beha/) of the _Heliand_ (using M, i.e. [Munich, Bayerische Staatsbibliothek, Cgm 25](https://www.digitale-sammlungen.de/de/view/bsb00026305), as its base manuscript, though with heavy emendation). The edition used is presumably the fourth, from 1922, as that's the public domain edition of record. The encoder has added puncti to represent caesuras, and the text has not been proofread, witness the form "atquepraeclaro" in the preface.

This notebook strips the HTML file of its tags and outputs a clean text to JSON and plaintext.

In [ ]:
import json
from mediawiki import MediaWiki
from pathlib import Path
from bs4 import BeautifulSoup

In [36]:
print_line_numbers = True
caesura_span = '    '
#caesura_span = '\t'

In [18]:
wikisource = MediaWiki(url='https://secure.wikimedia.org/wikisource/de/w/api.php')
page = wikisource.page('Heliand')
html = page.html
soup = BeautifulSoup(html, 'html.parser')
poem = soup.find('div', class_='poem')
plaintext = poem.get_text().split('\n')[1:]

In [ ]:
heliand = []
caesura = ' · '
for line in plaintext:
    if caesura in line:
        line = (line.split(' ', 1)[0], line.split(' ', 1)[1].split(caesura))
        heliand.append((int(line[0].rstrip('ab')), line[1][0], line[1][1].rstrip()))

In [31]:
json_file = 'heliand-m.json'
if not(Path(json_file).is_file()):
    with open(json_file, 'w', encoding='utf-8') as outfile:
        json.dump(heliand, outfile, ensure_ascii=False, indent=4)

In [37]:
plaintext_file = 'heliand-m.txt'
if Path(plaintext_file).is_file():
    with open(plaintext_file) as f:
        verse_lines = f.read().splitlines()
else:
    verse_lines = []
    for line in heliand:
        if print_line_numbers == True:
            reconstructed_line = str("{:04d}".format(int(line[0]))) + ' ' + line[1] + caesura_span + line[2]
        else:
            reconstructed_line = line[1] + caesura_span + line[2]
        verse_lines.append(reconstructed_line)
    with open(plaintext_file, 'w') as outfile:
        outfile.write('\n'.join(verse_lines))